In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

scenarios_dict_for_plots = ['results_001']

load_files_dict = {
    "results_001": {
        "folder_name" : "multisector_3zone",
        "results": "results_001/results",
        "base_path": "/Users/abbie/MacroEnergy-Abbie.jl/MacroEnergyExamples/examples"
    },
}
zones = ['SE', 'MIDAT','NE']

commodity_mapping = {
    "NaturalGas": "natgas",
    "Fossil_NaturalGas": "fossil_natgas",
    "Electricity": "elec",
    "Biomass_Agri": "bioagri",
    "Biomass_Herb": "bioherb",
    "Biomass_Wood": "biowood",
    "Biomass_Corn": "corn",
    "Ethanol": "ethanol",
    "Gasoline_E10": "gasoline_e10",
    "Gasoline": "gasoline",
    "Diesel": "diesel",
    "JetFuel": "jetfuel",
    "Fossil_JetFuel": "fossil_jetfuel",
    "Fossil_Gasoline": "fossil_gasoline",
    "Fossil_Gasoline_Blendstock": "fossil_gasoline_blendstock",
    "Fossil_Diesel": "fossil_diesel",
    "CO2Captured": "co2_captured",
    "Hydrogen": "h2"
}

multi_zone_mapping = ["co2_sink"]

# list of demand nodes that don't have a special fuel usage asset
final_demand_nodes_list = ['Electricity','Hydrogen']

# list of nodes that have a nsd as an option
nsd_nodes_list = ['Electricity','Hydrogen','NaturalGas','Gasoline','Diesel','JetFuel']

In [29]:
# manually change based on time weights
hours_per_year = 8760
weeks_per_year = 52
hours_per_rep_week = 168
scaling_factor = hours_per_year / (weeks_per_year * hours_per_rep_week)

total_hrs_in_rep_period = 168

## colors-mapping

In [30]:
# ------------- MANUALLY ASSIGN COLORS -------------
# colors based off of component_id
# component_id is used to distinguish "DryMill" from "DryMillCCS" using _commodity phrases
asset_colors = {
    "Electricity": {
        "Multiproduct_elec_edge": "plum",
        "Diesel_elec_edge": "plum",
        "Gasoline_elec_edge": "plum",

        "Ethanol_Bio_elec_edge": "plum",
        "Ethanol_BioCCS_20_elec_edge": "plum",
        "DryMillCCS_elec": "goldenrod",
        "DryMill_elec": "goldenrod",
        "DryMillCCS_RETROFIT_elec": "goldenrod",

        "natural_gas_fired_combined_cycle": "firebrick",
        "natural_gas_fired_combustion_turbine": "firebrick",
        "naturalgas_ccccsavgcf_conservative": "red",
        "naturalgas_ccavgcf": "red",
        "naturalgas_ctavgcf": "red",

        "utilitypv": "lemonchiffon",
        "existing_solar": "khaki",
        "solar_photovoltaic": "yellow",
        "existing_wind": "steelblue",
        "landbasedwind": "lightskyblue",
        "onshore_wind_turbine": "lightskyblue",
        "offshorewind_class10_moderate_floating_1_1_edge": "steelblue",

        "conventional_hydroelectric_1_discharge_edge": "lightseagreen",
        "pumpedhydro": "lightseagreen",
        "hydroelectric_pumped_storage_1_charge_edge": "lightseagreen",
        "small_hydroelectric_1_elec_edge": "lightseagreen",
        "hydroelectric_pumped_storage_1_discharge_edge": "paleturquoise",

        "Electricity_demand": "indigo",

        "battery": "mediumpurple",

        "Above_ground_storage_external_discharge_elec_edge" : "cadetblue",
        "Above_ground_storage_external_charge_elec_edge" : "cadetblue",
        "Above_ground_storage_discharge_elec_edge": "paleturquoise",
        "Above_ground_storage_charge_elec_edge": "paleturquoise",

        "SE_CCGT-H2_elec_edge" : "deepskyblue",
        "OCGT-H2_elec_edge" : "deepskyblue",
        "Electrolyzer_elec_edge": "paleturquoise",
        "CCGT-H2_elec_edge": "paleturquoise",
        "OCGT-H2_elec_edge": "paleturquoise",

        "nuclear_1_elec_edge" : "orange",
        "nuclear_2_elec_edge" : "orange",

        "Gasoline_Gasification_CCS_31_Herb_elec_consumption_edge" : "paleturquoise",
        "Gasoline_Gasification_CCS_31_Wood_elec_consumption_edge" : "paleturquoise",
        "Gasoline_Gasification_CCS_31_Agri_elec_consumption_edge" : "paleturquoise",
        "Gasoline_Gasification_CCS_99_Herb_elec_consumption_edge" : "paleturquoise",
        "Gasoline_Gasification_CCS_99_Wood_elec_consumption_edge" : "paleturquoise",
        "Gasoline_Gasification_CCS_99_Agri_elec_consumption_edge" : "paleturquoise",

        "Synfuel_Plant_elec_edge": "lightseagreen",

        "Large_SMR_wCCS_96pct_elec_edge": "paleturquoise",
        "Large_SMR_Non_CCS_elec_edge": "paleturquoise",
        "ATR_wCCS_94pct_elec_edge": "paleturquoise",

        "Bio_H2_Wood_elec_edge": "paleturquoise",
        "Bio_H2_Herb_elec_edge": "paleturquoise",
        "Bio_H2_Agri_elec_edge": "paleturquoise",
        "Bio_H2_CCS_99_Wood_elec_edge": "paleturquoise",
        "Bio_H2_CCS_99_Herb_elec_edge": "paleturquoise",
        "Bio_H2_CCS_99_Agri_elec_edge": "paleturquoise",

        "Bio_NG_CCS_40_Wood_elec_edge": "paleturquoise",
        "Bio_NG_CCS_40_Herb_elec_edge": "paleturquoise",
        "Bio_NG_CCS_40_Agri_elec_edge": "paleturquoise",
        "Bio_NG_CCS_99_Wood_elec_edge": "paleturquoise",
        "Bio_NG_CCS_99_Herb_elec_edge": "paleturquoise",
        "Bio_NG_CCS_99_Agri_elec_edge": "paleturquoise",

        "Bio_Electricity_CCS_93_Wood_elec_edge": "paleturquoise",
        "Bio_Electricity_CCS_93_Herb_elec_edge": "paleturquoise",
        "Bio_Electricity_CCS_93_Agri_elec_edge": "paleturquoise",

        "Solvent_wNGCC_DAC_elec_edge": "paleturquoise",
        "Sorbent_DAC_elec_edge": "paleturquoise",
        
    },
     "NaturalGas": {
        "Bio_H2_Wood_natgas_edge": "green",
        "Bio_H2_Herb_natgas_edge": "green",
        "Bio_H2_Agri_natgas_edge": "green",
        "Bio_H2_CCS_99_Wood_natgas_edge": "green",
        "Bio_H2_CCS_99_Herb_natgas_edge": "green",
        "Bio_H2_CCS_99_Agri_natgas_edge": "green",
        "Bio_H2_Agri_natgas_edge": "green",
        "Bio_H2_Wood_natgas_edge": "green",
        "Bio_H2_Herb_natgas_edge": "green",
    },
    "Diesel": {
        "Ethanol_Diesel_Upgrade_diesel_edge": "gold",
        "Ethanol_Multiproduct_Upgrade_diesel_edge": "#F4A300",
        "Diesel_Fossil_Upstream_fuel_edge": "black",
        "Diesel_Use_fuel_edge": "indigo",
        "Synfuel_Plant_diesel_edge": "magenta",

        "FT_High_Diesel_Non_CCS_Wood_diesel_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Wood_diesel_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Wood_diesel_edge": "#2F6B2F",

        "FT_High_Diesel_Non_CCS_Agri_diesel_edge": "#2F6B2F",
        "FT_High_Diesel_Agri_diesel_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Agri_diesel_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Agri_diesel_edge": "#2F6B2F",

        "FT_High_Diesel_Non_CCS_Herb_diesel_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Herb_diesel_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Herb_diesel_edge": "#2F6B2F",
    },
    "JetFuel": {
        "Ethanol_JetFuel_Upgrade_jetfuel_edge": "gold",
        "JetFuel_Fossil_Upstream_fuel_edge": "black",
        "JetFuel_Use_fuel_edge": "indigo",
        "Synfuel_Plant_jetfuel_edge": "magenta",

        "FT_High_Jetfuel_CCS_84_Wood_jetfuel_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_75_Wood_jetfuel_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Wood_jetfuel_edge": "#3AAFA9",

        "FT_High_Diesel_CCS_99_Wood_jetfuel_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Wood_jetfuel_edge": "#2F6B2F",
        "FT_High_Diesel_Non_CCS_Wood_jetfuel_edge": "#2F6B2F",

        "FT_High_Jetfuel_CCS_84_Herb_jetfuel_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_75_Herb_jetfuel_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Herb_jetfuel_edge": "#3AAFA9",

        "FT_High_Diesel_CCS_99_Herb_jetfuel_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Herb_jetfuel_edge": "#2F6B2F",
        "FT_High_Diesel_Non_CCS_Herb_jetfuel_edge": "#2F6B2F",

        "FT_High_Jetfuel_CCS_84_Agri_jetfuel_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_75_Agri_jetfuel_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Agri_jetfuel_edge": "#3AAFA9",
        
        "FT_High_Diesel_CCS_99_Agri_jetfuel_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Agri_jetfuel_edge": "#2F6B2F",
        "FT_High_Diesel_Non_CCS_Agri_jetfuel_edge": "#2F6B2F",
        "FT_High_Diesel_Agri_jetfuel_edge": "#2F6B2F",
    },

     "Gasoline": {
        "e10_gasoline_edge": "indigo",
        "Gasoline_Fossil_Upstream_fuel_edge": "black",
        "Ethanol_Gasoline_Upgrade_gasoline_edge": "gold",
        "Ethanol_Multiproduct_Upgrade_gasoline_edge": "#F4A300",
        "Gasoline_Use_fuel_edge": "indigo",
        "Synfuel_Plant_gasoline_edge": "magenta",
        
        "Gasoline_Gasification_Non_CCS_Agri_gasoline_edge": "#A4BF3B",
        "Gasoline_Gasification_CCS_31_Agri_gasoline_edge": "#A4BF3B",
        "Gasoline_Gasification_CCS_99_Agri_gasoline_edge": "#A4BF3B",

        "Gasoline_Gasification_Non_CCS_Herb_gasoline_edge": "#A4BF3B",
        "Gasoline_Gasification_CCS_31_Herb_gasoline_edge": "#A4BF3B",
        "Gasoline_Gasification_CCS_99_Herb_gasoline_edge": "#A4BF3B",

        "Gasoline_Gasification_Non_CCS_Wood_gasoline_edge": "#A4BF3B",
        "Gasoline_Gasification_CCS_31_Wood_gasoline_edge": "#A4BF3B",
        "Gasoline_Gasification_CCS_99_Wood_gasoline_edge": "#A4BF3B",

        "FT_High_Jetfuel_CCS_84_Wood_gasoline_edge": "#3AAFA9",
        "FT_High_Diesel_CCS_53_Wood_gasoline_edge": "#2F6B2F",
        "FT_High_Jetfuel_CCS_75_Wood_gasoline_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Wood_gasoline_edge": "#3AAFA9",
        "FT_High_Diesel_Non_CCS_Wood_gasoline_edge": "#2F6B2F",

        "FT_High_Jetfuel_CCS_84_Herb_gasoline_edge": "#3AAFA9",
        "FT_High_Diesel_CCS_53_Herb_gasoline_edge": "#2F6B2F",
        "FT_High_Jetfuel_CCS_75_Herb_gasoline_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Herb_gasoline_edge": "#3AAFA9",
        "FT_High_Diesel_Non_CCS_Herb_gasoline_edge": "#2F6B2F",

        "FT_High_Jetfuel_CCS_84_Agri_gasoline_edge": "#3AAFA9",
        "FT_High_Diesel_CCS_53_Agri_gasoline_edge": "#2F6B2F",
        "FT_High_Jetfuel_CCS_75_Agri_gasoline_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Agri_gasoline_edge": "#3AAFA9",
        "FT_High_Diesel_Non_CCS_Agri_gasoline_edge": "#2F6B2F",

        "FT_High_Diesel_Non_CCS_Wood_gasoline_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Wood_gasoline_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Wood_gasoline_edge": "#2F6B2F",

        "FT_High_Diesel_Agri_gasoline_edge": "#2F6B2F",
        "FT_High_Diesel_Non_CCS_Agri_gasoline_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Agri_gasoline_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Agri_gasoline_edge": "#2F6B2F",

        "FT_High_Diesel_Non_CCS_Herb_gasoline_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Herb_gasoline_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Herb_gasoline_edge": "#2F6B2F",
    },
    "Ethanol": {
        "DryMillCCS_86_ethanol": "darkorange",
        "DryMillCCS_20_ethanol": "darkorange",
        "DryMillCCS_ethanol": "orange",
        "DryMill_ethanol": "saddlebrown",
        "DryMillCCS_60_RETROFIT_ethanol": "goldenrod",
        "DryMillCCS_90_RETROFIT_ethanol": "gold",
        "Ethanol_Use_fuel": "indigo",
        "Ethanol_Bio_ethanol_edge": "orange",
        "Ethanol_BioCCS_ethanol_edge": "darkorange",
        "Ethanol_BioCCS_20_ethanol_edge": "darkorange",
        "Ethanol_BioCCS_80_ethanol_edge": "darkorange",
        "Ethanol_Multiproduct_Upgrade_ethanol_edge": "#B5C85A",
        "Ethanol_Diesel_Upgrade_ethanol_edge": "#2F6B2F",
        "e10_ethanol_edge": "indigo",
        "Ethanol_JetFuel_Upgrade_ethanol_edge": "#3AAFA9",
        "Ethanol_Gasoline_Upgrade_ethanol_edge": "#A4BF3B",
    },
    "NaturalGas" : {
        "DryMillCCS_natgas": "darkorange",
        "DryMill_natgas": "gold",
        "DryMillCCS_RETROFIT_natgas": "orange",
        "NG_Fossil_Upstream_fuel" : "black",
        "Sorbent_wNGCC_DAC" : "blue",
        "NG_End_Use_fuel_edge": "indigo",
        "Large_SMR_Non_CCS_fuel_edge": "deepskyblue",
        "Large_SMR_wCCS_96pct_fuel_edge": "deepskyblue",
        "ATR_wCCS_94pct_fuel_edge": "deepskyblue",
        "naturalgas_ccccsavgcf_conservative_0_fuel_edge": "red",
        "Syn_NG_Plant_natgas_edge": "magenta",
        "Synthetic_NaturalGas_natgas_edge": "deeppink",

        "DryMillCCS_60_RETROFIT_natgas_edge": "goldenrod",
        "DryMillCCS_90_RETROFIT_natgas_edge": "darkgoldenrod",

        "Solvent_DAC_natgas_edge": "#00A8A8",
        "Solvent_wNGCC_DAC_natgas_edge": "#00A8A8",

        "Bio_NG_Wood_natgas_edge": "green",
        "Bio_NG_Herb_natgas_edge": "green",
        "Bio_NG_Agri_natgas_edge": "green",
        "Bio_NG_CCS_40_Wood_natgas_edge": "green",
        "Bio_NG_CCS_40_Herb_natgas_edge": "green",
        "Bio_NG_CCS_40_Agri_natgas_edge": "green",
        "Bio_NG_CCS_99_Wood_natgas_edge": "green",
        "Bio_NG_CCS_99_Herb_natgas_edge": "green",
        "Bio_NG_CCS_99_Agri_natgas_edge": "green",

        "Bio_H2_Wood_natgas_edge": "green",
        "Bio_H2_Herb_natgas_edge": "green",
        "Bio_H2_Agri_natgas_edge": "green",
        "Bio_H2_CCS_99_Wood_natgas_edge": "green",
        "Bio_H2_CCS_99_Herb_natgas_edge": "green",
        "Bio_H2_CCS_99_Agri_natgas_edge": "green",

    },
    "CO2" : {
        "DryMillCCS_60_RETROFIT_co2_edge": "gold",
        "DryMillCCS_90_RETROFIT_co2_edge": "gold",
        "DryMillCCS_60_RETROFIT_co2_emission_edge": "gold",
        "DryMillCCS_90_RETROFIT_co2_emission_edge": "gold",
        "DryMillCCS_co2_edge": "gold",
        "DryMillCCS_co2_emission_edge" : "gold",
        "DryMill_co2_edge" : "gold",
        "DryMill_co2_emission_edge" : "gold",
        "Ethanol_BioCCS_80_co2_emission_edge" : "gold",
        "Ethanol_BioCCS_20_co2_emission_edge" : "gold",
        "Ethanol_BioCCS_20_co2_edge" : "gold",
        "Ethanol_BioCCS_80_co2_edge" : "gold",

        "Ethanol_Use_co2_edge" : "gold",

        "Gasoline_Gasification_CCS_31_Herb_co2_edge" : "#A4BF3B",
        "Gasoline_Gasification_CCS_31_Herb_co2_emission_edge" : "#A4BF3B",
        "Gasoline_Gasification_CCS_99_Herb_co2_edge" : "#A4BF3B",
        "Gasoline_Gasification_CCS_99_Herb_co2_emission_edge" : "#A4BF3B",
        "Gasoline_Gasification_Non_CCS_Herb_co2_edge": "#A4BF3B",
        "Gasoline_Gasification_Non_CCS_Herb_co2_emission_edge": "#A4BF3B",

        "Gasoline_Gasification_CCS_31_Agri_co2_edge" : "#A4BF3B",
        "Gasoline_Gasification_CCS_31_Agri_co2_emission_edge" : "#A4BF3B",
        "Gasoline_Gasification_CCS_99_Agri_co2_edge" : "#A4BF3B",
        "Gasoline_Gasification_CCS_99_Agri_co2_emission_edge" : "#A4BF3B",
        "Gasoline_Gasification_Non_CCS_Agri_co2_edge": "#A4BF3B",
        "Gasoline_Gasification_Non_CCS_Agri_co2_emission_edge": "#A4BF3B",

        "Gasoline_Gasification_CCS_31_Wood_co2_edge" : "#A4BF3B",
        "Gasoline_Gasification_CCS_31_Wood_co2_emission_edge" : "#A4BF3B",
        "Gasoline_Gasification_CCS_99_Wood_co2_edge" : "#A4BF3B",
        "Gasoline_Gasification_CCS_99_Wood_co2_emission_edge" : "#A4BF3B",
        "Gasoline_Gasification_Non_CCS_Wood_co2_edge": "#A4BF3B",
        "Gasoline_Gasification_Non_CCS_Wood_co2_emission_edge": "#A4BF3B",

        "Bio_H2_Wood_co2_emission_edge" : "turquoise",
        "Bio_H2_Herb_co2_emission_edge" : "turquoise",
        "Bio_H2_Agri_co2_emission_edge" : "turquoise",
        "Bio_H2_Wood_co2_edge" : "turquoise",
        "Bio_H2_Herb_co2_edge" : "turquoise",
        "Bio_H2_Agri_co2_edge" : "turquoise",
        "Bio_H2_CCS_99_Wood_co2_emission_edge" : "turquoise",
        "Bio_H2_CCS_99_Herb_co2_emission_edge" : "turquoise",
        "Bio_H2_CCS_99_Agri_co2_emission_edge" : "turquoise",
        "Bio_H2_CCS_99_Wood_co2_edge" : "turquoise",
        "Bio_H2_CCS_99_Herb_co2_edge" : "turquoise",
        "Bio_H2_CCS_99_Agri_co2_edge" : "turquoise",
        "ATR_wCCS_94pct_co2_edge" : "turquoise",

        "Bio_Electricity_CCS_93_Wood_co2_edge" : "lightblue",
        "Bio_Electricity_CCS_93_Herb_co2_edge" : "lightblue",
        "Bio_Electricity_CCS_93_Agri_co2_edge" : "lightblue",
        "Bio_Electricity_CCS_93_Wood_co2_emission_edge" : "lightblue",
        "Bio_Electricity_CCS_93_Herb_co2_emission_edge" : "lightblue",
        "Bio_Electricity_CCS_93_Agri_co2_emission_edge" : "lightblue",

        "Solvent_wNGCC_DAC_co2_emission_edge" : "#00A8A8",
        "Solvent_wNGCC_DAC_co2_edge" : "#00A8A8",
        "Sorbent_wNGCC_DAC_co2_edge" : "#00A8A8",
        "Sorbent_DAC_co2_edge" : "#00A8A8",

        "Bio_NG_CCS_40_Wood_co2_edge" : "crimson",
        "Bio_NG_CCS_40_Herb_co2_edge" : "crimson",
        "Bio_NG_CCS_40_Agri_co2_edge" : "crimson",
        "Bio_NG_CCS_40_Wood_co2_emission_edge" : "crimson",
        "Bio_NG_CCS_40_Herb_co2_emission_edge" : "crimson",
        "Bio_NG_CCS_40_Agri_co2_emission_edge" : "crimson",
        "Bio_NG_Wood_co2_edge" : "crimson",
        "Bio_NG_Herb_co2_edge" : "crimson",
        "Bio_NG_Agri_co2_edge" : "crimson",
        "Bio_NG_Wood_co2_emission_edge" : "crimson",
        "Bio_NG_Herb_co2_emission_edge" : "crimson",
        "Bio_NG_Agri_co2_emission_edge" : "crimson",
        "Bio_NG_CCS_99_Wood_co2_edge" : "crimson",
        "Bio_NG_CCS_99_Herb_co2_edge" : "crimson",
        "Bio_NG_CCS_99_Agri_co2_edge" : "crimson",
        "Bio_NG_CCS_99_Wood_co2_emission_edge" : "crimson",
        "Bio_NG_CCS_99_Herb_co2_emission_edge" : "crimson",
        "Bio_NG_CCS_99_Agri_co2_emission_edge" : "crimson",

        "JetFuel_Use_co2_edge" : "#3AAFA9",
        "Diesel_Use_co2_edge" : "#2F6B2F",
        "Gasoline_E10_Use_co2_edge" : "#A4BF3B",

        "Diesel_co2_emission_edge" : "#2F6B2F",
        "JetFuel_co2_emission_edge" : "#3AAFA9",
        "Gasoline_co2_emission_edge" : "#A4BF3B",
        "Multiproduct_co2_emission_edge" : "#A4BF3B",

        "NG_End_Use_co2_edge" : "crimson",

        "Synfuel_Plant_co2_emission_edge" : "magenta",
        "Large_SMR_wCCS_96pct_co2_edge": "turquoise",
        "Large_SMR_Non_CCS_co2_edge": "turquoise",

        "FT_High_Jetfuel_CCS_84_Wood_co2_edge": "#3AAFA9",
        "FT_High_Diesel_CCS_53_Wood_co2_edge": "#2F6B2F",
        "FT_High_Jetfuel_CCS_75_Wood_co2_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Wood_co2_edge": "#3AAFA9",
        "FT_High_Diesel_Non_CCS_Wood_co2_edge": "#2F6B2F",

        "FT_High_Jetfuel_CCS_84_Herb_co2_edge": "#3AAFA9",
        "FT_High_Diesel_CCS_53_Herb_co2_edge": "#2F6B2F",
        "FT_High_Jetfuel_CCS_75_Herb_co2_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Herb_co2_edge": "#3AAFA9",
        "FT_High_Diesel_Non_CCS_Herb_co2_edge": "#2F6B2F",

        "FT_High_Jetfuel_CCS_84_Agri_co2_edge": "#3AAFA9",
        "FT_High_Diesel_CCS_53_Agri_co2_edge": "#2F6B2F",
        "FT_High_Jetfuel_CCS_75_Agri_co2_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Agri_co2_edge": "#3AAFA9",
        "FT_High_Diesel_Non_CCS_Agri_co2_edge": "#2F6B2F",

        "FT_High_Diesel_Non_CCS_Wood_co2_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Wood_co2_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Wood_co2_edge": "#2F6B2F",

        "FT_High_Diesel_Agri_co2_edge": "#2F6B2F",
        "FT_High_Diesel_Non_CCS_Agri_co2_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Agri_co2_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Agri_co2_edge": "#2F6B2F",

        "FT_High_Diesel_Non_CCS_Herb_co2_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Herb_co2_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Herb_co2_edge": "#2F6B2F",

        "FT_High_Jetfuel_CCS_84_Wood_co2_emission_edge": "#3AAFA9",
        "FT_High_Diesel_CCS_53_Wood_co2_emission_edge": "#2F6B2F",
        "FT_High_Jetfuel_CCS_75_Wood_co2_emission_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Wood_co2_emission_edge": "#3AAFA9",
        "FT_High_Diesel_Non_CCS_Wood_co2_emission_edge": "#2F6B2F",

        "FT_High_Jetfuel_CCS_84_Herb_co2_emission_edge": "#3AAFA9",
        "FT_High_Diesel_CCS_53_Herb_co2_emission_edge": "#2F6B2F",
        "FT_High_Jetfuel_CCS_75_Herb_co2_emission_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Herb_co2_emission_edge": "#3AAFA9",
        "FT_High_Diesel_Non_CCS_Herb_co2_emission_edge": "#2F6B2F",

        "FT_High_Jetfuel_CCS_84_Agri_co2_emission_edge": "#3AAFA9",
        "FT_High_Diesel_CCS_53_Agri_co2_emission_edge": "#2F6B2F",
        "FT_High_Jetfuel_CCS_75_Agri_co2_emission_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Agri_co2_emission_edge": "#3AAFA9",
        "FT_High_Diesel_Non_CCS_Agri_co2_emission_edge": "#2F6B2F",

        "FT_High_Diesel_Non_CCS_Wood_co2_emission_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Wood_co2_emission_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Wood_co2_emission_edge": "#2F6B2F",

        "FT_High_Diesel_Agri_co2_emission_edge": "#2F6B2F",
        "FT_High_Diesel_Non_CCS_Agri_co2_emission_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Agri_co2_emission_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Agri_co2_emission_edge": "#2F6B2F",

        "FT_High_Diesel_Non_CCS_Herb_co2_emission_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Herb_co2_emission_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Herb_co2_emission_edge": "#2F6B2F",
        
    },
    "Hydrogen" : {
        "Electrolyzer_h2_edge": "paleturquoise",
        "Hydrogen_demand_edge" : "indigo",
        "Above_ground_storage_external_discharge_edge" : "cadetblue",
        "Above_ground_storage_external_charge_edge" : "cadetblue",
        "CCGT-H2_fuel_edge" : "deepskyblue",
        "OCGT-H2_fuel_edge" : "deepskyblue",
        "Large_SMR_Non_CCS_h2_edge" : "darkslategray",
        "Bio_H2_Herb_h2_edge" : "green",
        "Bio_H2_Agri_h2_edge" : "green",
        "Bio_H2_Wood_h2_edge" : "green",
        "ATR_wCCS_94pct_h2_edge" : "deepskyblue",
        "Large_SMR_wCCS_96pct_h2_edge" : "deepskyblue",

        "Bio_H2_CCS_99_Agri_h2_edge" : "green",
        "Bio_H2_CCS_99_Herb_h2_edge" : "green",
        "Bio_H2_CCS_99_Wood_h2_edge" : "green",

        "Synfuel_Plant_h2_edge" : "magenta",
        "Syn_NG_Plant_h2" : "magenta",

        "Diesel_h2_edge" : "green",

        "Ethanol_JetFuel_Upgrade_h2_edge": "dodgerblue",
    },
    "Biomass_Agri" : {
        "Gasoline_Gasification_Non_CCS_Agri_biomass_edge": "#A4BF3B",
        "Gasoline_Gasification_CCS_31_Agri_biomass_edge": "#A4BF3B",
        "Gasoline_Gasification_CCS_99_Agri_biomass_edge": "#A4BF3B",

        "FT_High_Jetfuel_CCS_84_Agri_biomass_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_75_Agri_biomass_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Agri_biomass_edge": "#3AAFA9",

        "FT_High_Diesel_Agri_biomass_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Agri_biomass_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Agri_biomass_edge": "#2F6B2F",

        "Ethanol_Bio_biomass_edge": "darkorange",
        "Ethanol_BioCCS_80_biomass_edge": "darkorange",
        "Ethanol_BioCCS_20_biomass_edge": "darkorange",

        "Bio_H2_Agri_biomass_edge": "turquoise",
        "Bio_H2_CCS_99_Agri_biomass_edge": "turquoise",

        "Bio_NG_Agri_biomass_edge": "crimson",
        "Bio_NG_CCS_40_Agri_biomass_edge": "crimson",
        "Bio_NG_CCS_99_Agri_biomass_edge": "crimson",

        "Bio_Electricity_CCS_93_Agri_biomass_edge": "lightblue"
    },
    "Biomass_Wood" : {
        "Gasoline_Gasification_Non_CCS_Wood_biomass_edge": "#A4BF3B",
        "Gasoline_Gasification_CCS_31_Wood_biomass_edge": "#A4BF3B",
        "Gasoline_Gasification_CCS_99_Wood_biomass_edge": "#A4BF3B",

        "FT_High_Jetfuel_CCS_84_Wood_biomass_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_75_Wood_biomass_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Wood_biomass_edge": "#3AAFA9",

        "FT_High_Diesel_Wood_biomass_edge": "#2F6B2F",
        "FT_High_Diesel_Non_CCS_Wood_biomass_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Wood_biomass_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Wood_biomass_edge": "#2F6B2F",

        "Bio_H2_Wood_biomass_edge": "turquoise",
        "Bio_H2_CCS_99_Wood_biomass_edge": "turquoise",

        "Bio_NG_Wood_biomass_edge": "crimson",
        "Bio_NG_CCS_40_Wood_biomass_edge": "crimson",
        "Bio_NG_CCS_99_Wood_biomass_edge": "crimson",

        "Bio_Electricity_CCS_93_Wood_biomass_edge": "lightblue"
    },
    "Biomass_Herb" : {
        "Gasoline_Gasification_Non_CCS_Herb_biomass_edge": "#A4BF3B",
        "Gasoline_Gasification_CCS_31_Herb_biomass_edge": "#A4BF3B",
        "Gasoline_Gasification_CCS_99_Herb_biomass_edge": "#A4BF3B",

        "FT_High_Jetfuel_CCS_84_Herb_biomass_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_75_Herb_biomass_edge": "#3AAFA9",
        "FT_High_Jetfuel_CCS_99_Herb_biomass_edge": "#3AAFA9",

        "FT_High_Diesel_Herb_biomass_edge": "#2F6B2F",
        "FT_High_Diesel_Non_CCS_Herb_biomass_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_53_Herb_biomass_edge": "#2F6B2F",
        "FT_High_Diesel_CCS_99_Herb_biomass_edge": "#2F6B2F",

        "Bio_H2_Herb_biomass_edge": "turquoise",
        "Bio_H2_CCS_99_Herb_biomass_edge": "turquoise",

        "Bio_NG_Herb_biomass_edge": "crimson",
        "Bio_NG_CCS_40_Herb_biomass_edge": "crimson",
        "Bio_NG_CCS_99_Herb_biomass_edge": "crimson",

        "Bio_Electricity_CCS_93_Herb_biomass_edge": "lightblue"
    },
    "General": {
        "_to_": "gray",
        "nsd": "gray"
    }
}

## prepare-plots

In [31]:
# ------------- INITIALIZE NESTED DICTIONARY FOR EACH SCENARIO -------------
categories = ["flow_total","flow_by_commodity", "capacity", "lcoe", "commodities_list", "assets_list","demand","nsd","duals","costs","capacity","time_weights"]
scenario_plot_dict = {
    scenario_label: {category: {} for category in categories}
    for scenario_label in load_files_dict
}

def summarize_data_structure(data):
    for scenario, categories in data.items():
        print(f"\n📘 Scenario: {scenario}")
        for category, content in categories.items():
            if isinstance(content, dict) and content:
                print(f"  ├─ {category}:")
                for key, value in content.items():
                    if isinstance(value, dict):
                        print(f"  │   • {key} ({len(value)} sub-items)")
                    elif isinstance(value, list):
                        if all(isinstance(x, str) for x in value):
                            preview = ', '.join(value[:3]) + ("..." if len(value) > 3 else "")
                            print(f"  │   • {key} → list of {len(value)} strings [{preview}]")
                        else:
                            print(f"  │   • {key} → list of {len(value)} items")
                    elif hasattr(value, 'shape'):
                        print(f"  │   • {key} → DataFrame {value.shape}")
                    else:
                        print(f"  │   • {key} → {type(value).__name__}")
            elif isinstance(content, list) and content:
                print(f"  ├─ {category}: list of {len(content)} items")
            elif hasattr(content, 'shape'):
                print(f"  ├─ {category}: DataFrame {content.shape}")
            else:
                print(f"  ├─ {category}: (empty)")

                
summarize_data_structure(scenario_plot_dict)


📘 Scenario: results_001
  ├─ flow_total: (empty)
  ├─ flow_by_commodity: (empty)
  ├─ capacity: (empty)
  ├─ lcoe: (empty)
  ├─ commodities_list: (empty)
  ├─ assets_list: (empty)
  ├─ demand: (empty)
  ├─ nsd: (empty)
  ├─ duals: (empty)
  ├─ costs: (empty)
  ├─ time_weights: (empty)


In [32]:
# ------------- Construct time weights FOR EACH SCENARIO using manually provided info + Period_map.csv -------------
for scenario_label in load_files_dict.keys():
    print("scenario_label: ", scenario_label)
    base_path = Path(load_files_dict[scenario_label]["base_path"])
    folder_name = load_files_dict[scenario_label]["folder_name"]
    results = load_files_dict[scenario_label]["results"]

    path = base_path / folder_name / "system" / "Period_map.csv"

    rep_week_map_df = pd.read_csv(path)
    rep_week_counts = rep_week_map_df["Rep_Period_Index"].value_counts().sort_index() # what does this do?

    time_weights = [count * scaling_factor for count in rep_week_counts]
    full_time_weights = [w for w in time_weights for _ in range(hours_per_rep_week)]
    print("full_time_weights: ", full_time_weights) # a weight for each of the hours in the sample sub period mapping to the annual?

    scenario_plot_dict[scenario_label]["time_weights"] = full_time_weights

scenario_label:  results_001
full_time_weights:  [23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.063186813186814, 23.0631868

In [33]:
# ------------- Add LIST OF COMMODITIES FOR EACH SCENARIO using flow.csv -------------
for scenario_label in load_files_dict.keys():
    base_path = Path(load_files_dict[scenario_label]["base_path"])
    folder_name = load_files_dict[scenario_label]["folder_name"]
    results = load_files_dict[scenario_label]["results"]
    
    path = base_path / folder_name / results / "flows.csv"

    # Load file and extract unique commodities
    df = pd.read_csv(path)
    commodities = df["commodity"].astype(str).unique().tolist()

    # Save to data dictionary
    scenario_plot_dict[scenario_label]["commodities_list"] = commodities
    print(commodities)

summarize_data_structure(scenario_plot_dict)

['Biomass_Herb', 'Electricity', 'CO2', 'CO2Captured', 'Biomass_Wood', 'Gasoline', 'Hydrogen', 'JetFuel', 'Diesel', 'NaturalGas', 'Fossil_Gasoline', 'Fossil_JetFuel', 'Fossil_Diesel', 'Fossil_NaturalGas', 'Uranium']

📘 Scenario: results_001
  ├─ flow_total: (empty)
  ├─ flow_by_commodity: (empty)
  ├─ capacity: (empty)
  ├─ lcoe: (empty)
  ├─ commodities_list: list of 15 items
  ├─ assets_list: (empty)
  ├─ demand: (empty)
  ├─ nsd: (empty)
  ├─ duals: (empty)
  ├─ costs: (empty)
  ├─ time_weights: list of 504 items


In [34]:
# ------------- Add full LIST OF ASSET ID'S FOR EACH SCENARIO using flows.csv -------------
for scenario_label in load_files_dict.keys():
    base_path = Path(load_files_dict[scenario_label]["base_path"])
    folder_name = load_files_dict[scenario_label]["folder_name"]
    results = load_files_dict[scenario_label]["results"]
    
    path = base_path / folder_name / results / "flows.csv"

    # Load file and extract unique commodities
    df = pd.read_csv(path)
    commodities = df["resource_id"].astype(str).unique().tolist()

    # Save to data dictionary
    scenario_plot_dict[scenario_label]["assets_list"] = commodities

summarize_data_structure(scenario_plot_dict)


📘 Scenario: results_001
  ├─ flow_total: (empty)
  ├─ flow_by_commodity: (empty)
  ├─ capacity: (empty)
  ├─ lcoe: (empty)
  ├─ commodities_list: list of 15 items
  ├─ assets_list: list of 215 items
  ├─ demand: (empty)
  ├─ nsd: (empty)
  ├─ duals: (empty)
  ├─ costs: (empty)
  ├─ time_weights: list of 504 items


In [35]:
# ------------- add the demand csv for the sample time period for each scenario using demand fuel.csv -------------
for scenario_label in load_files_dict.keys():
    base_path = Path(load_files_dict[scenario_label]["base_path"])
    folder_name = load_files_dict[scenario_label]["folder_name"]
    results = load_files_dict[scenario_label]["results"]
    #time_weights = scenario_plot_dict[scenario_label]["time_weights"]
    
    path = base_path / folder_name / "system" / "demand fuel.csv"

    # Load file and extract unique commodities
    df = pd.read_csv(path)

    df_sums = pd.DataFrame(df.sum()).T
    df_sums['Time_Index'] = len(df)
    display(df_sums)

    scenario_plot_dict[scenario_label]["demand"] = df_sums
    
summarize_data_structure(scenario_plot_dict)

,Time_Index,Demand_MW_z1,Demand_MW_z2,Demand_MW_z3,Demand_H2_z1,Demand_H2_z2,Demand_H2_z3,Demand_NG_z1,Demand_NG_z2,Demand_NG_z3,Demand_Gasoline_z1,Demand_Gasoline_z2,Demand_Gasoline_z3,Demand_JetFuel_z1,Demand_JetFuel_z2,Demand_JetFuel_z3,Demand_Diesel_z1,Demand_Diesel_z2,Demand_Diesel_z3,Demand_Zero
0,504,6.319043e+07,4.753489e+07,2.512638e+07,7444710.0,3712428.72,913217.76,1.196551e+07,1.476147e+07,1.051089e+07,2.191496e+07,1.375189e+07,7877403.828,8.757595e+06,4.791936e+06,4.966527e+06,1.028238e+07,6.609628e+06,4.176142e+06,0.0



📘 Scenario: results_001
  ├─ flow_total: (empty)
  ├─ flow_by_commodity: (empty)
  ├─ capacity: (empty)
  ├─ lcoe: (empty)
  ├─ commodities_list: list of 15 items
  ├─ assets_list: list of 215 items
  ├─ demand: DataFrame (1, 20)
  ├─ nsd: (empty)
  ├─ duals: (empty)
  ├─ costs: (empty)
  ├─ time_weights: list of 504 items


In [36]:
# ------------- add the NSD for the sampled time period for each scenario with nse.csv from get_results.jl -------------
for scenario_label in load_files_dict.keys():
    base_path = Path(load_files_dict[scenario_label]["base_path"])
    folder_name = load_files_dict[scenario_label]["folder_name"]
    results = load_files_dict[scenario_label]["results"]
    #time_weights = scenario_plot_dict[scenario_label]["time_weights"]
    
    path = base_path / folder_name / results / "nse.csv"

    # Load file and extract unique commodities
    df = pd.read_csv(path)

    df_sums = pd.DataFrame(df.sum()).T
    df_sums['time'] = len(df)
    display(df_sums)

    # ✅ Save inside the loop (so each scenario gets saved properly)
    scenario_plot_dict[scenario_label]["nsd"] = df_sums
    
summarize_data_structure(scenario_plot_dict)

,time,elec_MIDAT,elec_NE,elec_SE,h2_MIDAT,h2_NE,h2_SE
0,504,0.0,0.0,0.0,0.0,0.0,0.0



📘 Scenario: results_001
  ├─ flow_total: (empty)
  ├─ flow_by_commodity: (empty)
  ├─ capacity: (empty)
  ├─ lcoe: (empty)
  ├─ commodities_list: list of 15 items
  ├─ assets_list: list of 215 items
  ├─ demand: DataFrame (1, 20)
  ├─ nsd: DataFrame (1, 7)
  ├─ duals: (empty)
  ├─ costs: (empty)
  ├─ time_weights: list of 504 items


In [37]:
# ------------- add the DISCOUNTED COSTS for each scenario with costs.csv -------------
for scenario_label in load_files_dict.keys():
    base_path = Path(load_files_dict[scenario_label]["base_path"])
    folder_name = load_files_dict[scenario_label]["folder_name"]
    results = load_files_dict[scenario_label]["results"]
    
    path = base_path / folder_name / results / "costs.csv"
    df = pd.read_csv(path)
    scenario_plot_dict[scenario_label]["costs"] = df
    
summarize_data_structure(scenario_plot_dict)


📘 Scenario: results_001
  ├─ flow_total: (empty)
  ├─ flow_by_commodity: (empty)
  ├─ capacity: (empty)
  ├─ lcoe: (empty)
  ├─ commodities_list: list of 15 items
  ├─ assets_list: list of 215 items
  ├─ demand: DataFrame (1, 20)
  ├─ nsd: DataFrame (1, 7)
  ├─ duals: (empty)
  ├─ costs: DataFrame (3, 10)
  ├─ time_weights: list of 504 items


In [38]:
# ------------- add the duals for the sample time period for each scenario using balance_duals.csv -------------
for scenario_label in load_files_dict.keys():
    base_path = Path(load_files_dict[scenario_label]["base_path"])
    folder_name = load_files_dict[scenario_label]["folder_name"]
    results = load_files_dict[scenario_label]["results"]
    
    path = base_path / folder_name / results / "balance_duals.csv"

    # Load file and extract unique commodities
    df = pd.read_csv(path)

    # Compute mean of non-zero values for each column
    #df_means = df.apply(lambda col: col[col != 0].mean() if (col != 0).any() else 0)
    df_means = df.mean()
    df_means = pd.DataFrame([df_means])  # Convert back to single-row DataFrame

    df_means['time'] = len(df_means)
    display(df_means)

    scenario_plot_dict[scenario_label]["duals"] = df_means
    
summarize_data_structure(scenario_plot_dict)

,natgas_SE,natgas_MIDAT,natgas_NE,natgas_demand_SE,natgas_demand_MIDAT,natgas_demand_NE,biowood_SE,biowood_MIDAT,biowood_NE,bioherb_SE,...,jetfuel_demand_SE,jetfuel_demand_MIDAT,jetfuel_demand_NE,diesel_SE,diesel_MIDAT,diesel_NE,diesel_demand_SE,diesel_demand_MIDAT,diesel_demand_NE,time
0,-46.464541,-45.169903,-26.427722,162.400562,163.353986,184.894123,688.499865,674.989146,1100.992955,683.780673,...,138.922887,130.318484,342.802898,97.34841,97.34841,97.34841,371.499514,371.499514,371.499514,1



📘 Scenario: results_001
  ├─ flow_total: (empty)
  ├─ flow_by_commodity: (empty)
  ├─ capacity: (empty)
  ├─ lcoe: (empty)
  ├─ commodities_list: list of 15 items
  ├─ assets_list: list of 215 items
  ├─ demand: DataFrame (1, 20)
  ├─ nsd: DataFrame (1, 7)
  ├─ duals: DataFrame (1, 40)
  ├─ costs: DataFrame (3, 10)
  ├─ time_weights: list of 504 items


In [39]:
# ------------- add capacity for each scenario using capacity.csv -------------
for scenario_label in load_files_dict.keys():
    base_path = Path(load_files_dict[scenario_label]["base_path"])
    folder_name = load_files_dict[scenario_label]["folder_name"]
    results = load_files_dict[scenario_label]["results"]
    
    path = base_path / folder_name / results / "capacity.csv"

    # Load file and extract unique commodities
    df = pd.read_csv(path)

    scenario_plot_dict[scenario_label]["capacity"] = df
    
summarize_data_structure(scenario_plot_dict)


📘 Scenario: results_001
  ├─ flow_total: (empty)
  ├─ flow_by_commodity: (empty)
  ├─ capacity: DataFrame (633, 8)
  ├─ lcoe: (empty)
  ├─ commodities_list: list of 15 items
  ├─ assets_list: list of 215 items
  ├─ demand: DataFrame (1, 20)
  ├─ nsd: DataFrame (1, 7)
  ├─ duals: DataFrame (1, 40)
  ├─ costs: DataFrame (3, 10)
  ├─ time_weights: list of 504 items


In [45]:
# ------------- combine time weights and flow.csv results to make flow results that represent the full period from sample period -------------
for scenario_label in load_files_dict.keys():
    # --- Load file paths ---
    base_path = Path(load_files_dict[scenario_label]["base_path"])
    folder_name = load_files_dict[scenario_label]["folder_name"]
    results = load_files_dict[scenario_label]["results"]
    path = base_path / folder_name / results / "flows.csv"

    # --- Read the flow file ---
    flow_df = pd.read_csv(path)
    #display(flow_df)

    # --- Get the time weights for this scenario ---
    full_time_weights = scenario_plot_dict[scenario_label]["time_weights"]

    # --- Prepare an empty list to collect processed dataframes ---
    weighted_list = []

    # --- Loop through each unique (commodity, component_id) combination ---
    for (commodity, component_id), df_for_one_commodity in flow_df.groupby(["commodity", "component_id"]):
        df_for_one_commodity = df_for_one_commodity.sort_values("time").reset_index(drop=True)

        # --- Check if the time steps match expectation ---
        #if float(len(df_for_one_commodity)) != float(total_hrs_in_rep_period):
        #    print(f"Skipping {commodity}-{component_id} — expected 504 time steps, got {len(df_for_one_commodity)}.")
        #    continue

        # --- Add weight and weighted value columns ---
        df_for_one_commodity["time_weight"] = full_time_weights
        #display(df_for_one_commodity)
        df_for_one_commodity["weighted_value"] = (
            df_for_one_commodity["value"] * df_for_one_commodity["time_weight"]
        )

        # --- Store for later concatenation ---
        weighted_list.append(df_for_one_commodity)

    # --- Combine all weighted data back into one dataframe ---
    weighted_flows = pd.concat(weighted_list, ignore_index=True)

    scenario_plot_dict[scenario_label]["flow_total"] = weighted_flows

    # --- Save to CSV to inspect ---
    out_path = base_path / folder_name / results / "flows_weighted.csv"
    weighted_flows.to_csv(out_path, index=False)

    print(f"✅ Saved weighted flows for {scenario_label} to {out_path}")

369432
✅ Saved weighted flows for results_001 to /Users/abbie/MacroEnergy-Abbie.jl/MacroEnergyExamples/examples/multisector_3zone/results_001/results/flows_weighted.csv


## plot-code

In [ ]:
def strip_zone_prefix(label, zones):
    if label.endswith("_edge"):
        label = label[:-5]
    for z in zones:
        prefix = z + "_"     # turn 'NCEN' → 'NCEN_'
        if label.startswith(prefix):
            return label[len(prefix):]   # strip it
    return label

In [ ]:
def plot_scenario_balance_horizontal_plotly(
    data, commodity, scenarios,
    units="MWh", sort_components=True,
    figsize=(550, 300),
    title=None, show=True, legend=True,
):
    """
    Plotly version of plot_scenario_balance_horizontal.
    Keeps all data logic, color logic, sorting, and conditionals unchanged.
    """

    # -------- 1. Load + concat --------
    frames = []
    for scen in scenarios:
        df = data[scen]["flow_by_commodity"][commodity].copy()
        df["scenario"] = scen
        frames.append(df)

    df = pd.concat(frames, ignore_index=True)
    df["weighted_value_flow"] = (
        pd.to_numeric(df["weighted_value_flow"], errors="coerce").fillna(0)
    )

    # Convert MWh → EJ
    if commodity not in ["Biomass_Agri", "Biomass_Herb", "Biomass_Wood"]:
        df["weighted_value_flow"] *= 3.6e-9

    # -------- 2. Aggregate --------
    agg = (
        df.groupby(["scenario", "component_id"], as_index=False)["weighted_value_flow"]
        .sum()
    )

    # -------- Print totals --------
    print("\n=== Scenario Totals ===")
    for scen in scenarios:
        df_s = agg[agg["scenario"] == scen]["weighted_value_flow"]
        total_pos = df_s[df_s > 0].sum()
        total_neg = df_s[df_s < 0].sum()
        print(f"{scen}:  +{total_pos:,.2f}   {total_neg:,.2f}")

    # -------- 3. Component color / sorting --------
    comps = agg["component_id"].unique().tolist()

    color_map = {}
    for comp in comps:
        assigned = False
        for sector, comp_dict in asset_colors.items():
            for cid, ccol in comp_dict.items():
                if cid in comp:
                    color_map[comp] = ccol
                    assigned = True
                    break
            if assigned: break
        if not assigned:
            color_map[comp] = "gray"

    comp_info = []
    for comp in comps:
        total_abs = abs(agg[agg["component_id"] == comp]["weighted_value_flow"].sum())
        comp_info.append((color_map.get(comp, "gray"), total_abs, comp))

    comp_info.sort(key=lambda x: (x[0], -x[1]))
    comps = [c[2] for c in comp_info]

    lookup = agg.set_index(["scenario", "component_id"])["weighted_value_flow"].to_dict()

    # -------- 4. Plotly plotting --------
    N = len(scenarios)
    left_neg = np.zeros(N)
    left_pos = np.zeros(N)

    fig = go.Figure()
    y_positions = np.arange(N)  # use numeric for stacking base

    # Negative bars
    for comp in comps:
        vals = np.array([lookup.get((s, comp), 0.0) for s in scenarios])
        neg_mask = vals < 0
        if np.any(neg_mask):
            fig.add_bar(
                y=np.array(scenarios)[neg_mask],
                x=vals[neg_mask],
                base=left_neg[neg_mask],
                orientation="h",
                marker_color=color_map[comp],
                name=strip_zone_prefix(comp, zones),
                hovertemplate="<b>Technology:</b> %{customdata}<br>"
                              "<b>Scenario:</b> %{y}<br>"
                              "<b>Value:</b> %{x:.3e}<extra></extra>",
                customdata=[comp] * np.sum(neg_mask),
            )
            left_neg[neg_mask] += vals[neg_mask]

    # Positive bars
    for comp in comps:
        vals = np.array([lookup.get((s, comp), 0.0) for s in scenarios])
        pos_mask = vals > 0
        if np.any(pos_mask):
            fig.add_bar(
                y=np.array(scenarios)[pos_mask],
                x=vals[pos_mask],
                base=left_pos[pos_mask],
                orientation="h",
                marker_color=color_map[comp],
                name=strip_zone_prefix(comp, zones),
                hovertemplate="<b>Technology:</b> %{customdata}<br>"
                              "<b>Scenario:</b> %{y}<br>"
                              "<b>Value:</b> %{x:.3e}<extra></extra>",
                customdata=[comp] * np.sum(pos_mask),
            )
            left_pos[pos_mask] += vals[pos_mask]

    max_extent = max(abs(left_neg.min()), abs(left_pos.max()))

    # Formatting like Matplotlib
    fig.update_layout(
        barmode="relative",
        width=figsize[0],
        height=figsize[1],
        title=title,
        xaxis=dict(
            zeroline=True,
            zerolinewidth=2,
            range=[-1.1 * max_extent, 1.1 * max_extent],
        ),
        yaxis=dict(
            tickfont=dict(size=22),
            categoryorder="array",
            categoryarray=scenarios,
        ),
        legend=dict(
            orientation="v",
            x=1.02,
            y=0.5,
            font=dict(size=14),
        ) if legend else dict(visible=False),
    )

    # Biomass vertical lines
    if commodity == "Biomass_Herb":
        fig.add_vline(x=-5.7e7, line_dash="dash", line_color="red")
    if commodity == "Biomass_Agri":
        fig.add_vline(x=-1.12e8, line_dash="dash", line_color="red")
    if commodity == "Biomass_Wood":
        fig.add_vline(x=-2.39e7, line_dash="dash", line_color="red")

    if show:
        fig.show()

    return fig